In [12]:
import mwclient
import tqdm
import requests

In [21]:
import mwclient
from tqdm import tqdm

site = mwclient.Site('wiki.hypixel.net', path='/')

# 1. Get all page titles (fast)
page_names = [page.name for page in site.allpages(namespace=0)]

# 2. Fetch full text with progress bar
all_pages = []
for title in tqdm(page_names, desc="Fetching text"):
    if len(all_pages) >= 100:
        break
    page = site.pages[title]
    all_pages.append({
        'title': title,
        'text': page.text(),
        'links': []  # will fill next
    })

# 3. Fill links (also with progress bar)
for entry in tqdm(all_pages, desc="Fetching links"):
    p = site.pages[entry['title']]
    entry['links'] = [
        link.name for link in p.links() 
        if link.namespace == 0   # only main articles
    ]

Fetching links: 100%|██████████| 100/100 [01:00<00:00,  1.65it/s]


In [22]:
all_pages

[{'title': '"Your" Island',
  'text': '{{Zone Page\n|zone = BACK_2_BASICS\n|requirements = {{SkyBlock Level|12}} <br> Secured the {{Item/RIFT_TROPHY_CHICKEN_N_EGG|is=15}} <br> Secured the {{Item/RIFT_TROPHY_MIRRORED|is=15}}\n|coordinates = -91.5, 20, -31.5\n|items = \n*{{Item/BEDWARS_WOOL|is=15}}\n|npcs = \n\n|summary = \'\'\'"Your" Island\'\'\' is a [[Location]] in the {{Zone/RIFT_DIMENSION}}. It looks very similar to the default [[Private Island]], and it can be entered from the center portal in the {{Zone/VILLAGE_PLAZA}}.\n\n|purpose = Players can chop down the oak tree to obtain {{Item/BEDWARS_WOOL}}, which can be placed to bridge across to the other platform where {{NPC/RIFT_JERRY}} can be found. Next to him is a portal back to the {{Zone/VILLAGE_PLAZA}}.\n\n|trivia = \n\n|alpha_history = {{Alpha Version\n|patch = 5354674\n|change1 = \'\'\'"Your" Island\'\'\' Added.\n}}\n\n|history = {{SkyBlock Version\n|patch = 5399251\n|change1 = \'\'\'"Your" Island\'\'\' Added.\n}}\n}}',
  'lin

In [23]:
import networkx as nx

G = nx.DiGraph()

# Add nodes with the page text as an attribute
for page in all_pages:
    G.add_node(page['title'], text=page['text'])

# Add edges for internal links (only if the target page is in our crawled set)
all_titles = set(page['title'] for page in all_pages)
for page in all_pages:
    source = page['title']
    for target in page['links']:
        if target in all_titles:          # link only to pages we actually have
            G.add_edge(source, target)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph built: 100 nodes, 1 edges
